<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session11_evaluating_and_publishing_models_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۱: ارزیابی و انتشار مدل فاین‌تیون‌شده

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session11_evaluating_and_publishing_models.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** آماده‌سازی این نوت‌بوک، در صورت نبود آداپتور، مدل را یک بار آموزش می‌دهد و آن را ادغام می‌کند (حدود ۱۵ دقیقه روی T4).

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install -U transformers trl peft bitsandbytes datasets huggingface_hub
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import json
import os
import re
import time

import pandas as pd
import torch
from datasets import Dataset
from llm_utils import download_data
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


def load_jsonl(name):
    with open(download_data(name), encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_data, test_data, real_data = (load_jsonl(f"orders_{s}.jsonl") for s in ("train", "test", "real"))
new_templates = test_data[150:]

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_SHORT = ("Extract the order from the Persian customer message as JSON with keys "
                "product_id, quantity, color, city.")
FIELDS = ["product_id", "quantity", "color", "city"]
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def conversations(data):
    return [[{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}] for d in data]


def generate(model, convs, batch_size=16, max_new_tokens=80):
    tokenizer.padding_side = "left"
    outputs = []
    for i in range(0, len(convs), batch_size):
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs[i:i + batch_size]]
        batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(out[:, batch.input_ids.shape[1]:], skip_special_tokens=True)
    return outputs


def parse(output):
    match = re.search(r"\{.*\}", output, re.S)
    try:
        return json.loads(match.group()) if match else None
    except json.JSONDecodeError:
        return None


def same(a, b):
    norm = lambda x: None if x in (None, "", "null") else str(x).replace("\u200c", " ").strip()
    return norm(a) == norm(b)


def score(outputs, data):
    rows = []
    for out, item in zip(outputs, data):
        pred = parse(out) or {}
        row = {"message": item["message"], "output": out.strip(), "valid_json": bool(pred)}
        row.update({f: bool(pred) and same(pred.get(f), item["order"][f]) for f in FIELDS})
        row["all_fields"] = all(row[f] for f in FIELDS)
        rows.append(row)
    return pd.DataFrame(rows)


def to_dataset(data):
    return Dataset.from_list([
        {"prompt": [{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}],
         "completion": [{"role": "assistant", "content": json.dumps(d["order"], ensure_ascii=False)}]}
        for d in data])


def train(data, r=16, output_dir="order-extractor-lora", epochs=2):
    """Train a QLoRA adapter and return (model, training seconds)."""
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
    tokenizer.padding_side = "right"
    trainer = SFTTrainer(
        model=model, train_dataset=to_dataset(data), processing_class=tokenizer,
        peft_config=LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules="all-linear",
                               task_type="CAUSAL_LM"),
        args=SFTConfig(output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=8,
                       gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                       warmup_steps=0.05, logging_steps=20, max_length=256, fp16=True, bf16=False,
                       save_strategy="no", report_to="none"))
    start = time.time()
    trainer.train()
    seconds = time.time() - start
    trainer.model.save_pretrained(output_dir)
    trainer.model.eval()
    return trainer.model, seconds


def free(*objects):
    for o in objects:
        del o
    torch.cuda.empty_cache()

In [ ]:
ADAPTER_DIR = "order-extractor-lora"
if not os.path.isdir(ADAPTER_DIR):
    model, _ = train(train_data, output_dir=ADAPTER_DIR)
    del model
    torch.cuda.empty_cache()

if not os.path.isdir("order-extractor-merged"):
    base_fp16 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="auto")
    merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR).merge_and_unload()
    merged.save_pretrained("order-extractor-merged")
    tokenizer.save_pretrained("order-extractor-merged")
    del merged, base_fp16
    torch.cuda.empty_cache()

if not os.path.isdir("llama.cpp"):
    os.system("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp")
    os.system("pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")

from llm_utils import setup_llm, start_ollama
start_ollama()

CHAT_TEMPLATE = ("{{- range .Messages }}<|im_start|>{{ .Role }}\n{{ .Content }}<|im_end|>\n"
                 "{{ end }}<|im_start|>assistant\n")


def create_ollama_model(gguf_path, name):
    with open(f"Modelfile.{name}", "w") as f:
        f.write(f'FROM ./{gguf_path}\nTEMPLATE """{CHAT_TEMPLATE}"""\n'
                'PARAMETER stop <|im_end|>\nPARAMETER temperature 0\n')
    os.system(f"ollama create {name} -f Modelfile.{name} > /dev/null")
    return setup_llm("ollama", model=name)


def ollama_outputs(client, model_name, data):
    return [client.chat.completions.create(model=model_name, messages=c, temperature=0).choices[0].message.content
            for c in conversations(data)]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

مدل را با `--outtype f16` هم به GGUF تبدیل کنید. حجم دو فایل را مقایسه کنید و بررسی کنید روی چند درصد پیام‌های آزمون، خروجی دو مدل یکسان است.

</div>

In [ ]:
for outtype in ["q8_0", "f16"]:
    if not os.path.exists(f"order-extractor-{outtype}.gguf"):
        os.system(f"python llama.cpp/convert_hf_to_gguf.py order-extractor-merged "
                  f"--outfile order-extractor-{outtype}.gguf --outtype {outtype} > /dev/null 2>&1")
    print(f"{outtype}: {os.path.getsize(f'order-extractor-{outtype}.gguf') / 1e9:.2f} GB")

sample = test_data[100:150] + new_templates[:50]
outputs = {}
for outtype in ["q8_0", "f16"]:
    client, name = create_ollama_model(f"order-extractor-{outtype}.gguf", f"order-extractor-{outtype.replace('_', '')}")
    outputs[outtype] = ollama_outputs(client, name, sample)

identical = sum(parse(a) == parse(b) for a, b in zip(outputs["q8_0"], outputs["f16"]))
print(f"identical parsed outputs: {identical}/{len(sample)}")
for outtype in outputs:
    print(outtype, "accuracy:", round(score(outputs[outtype], sample).all_fields.mean(), 3))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

فایل `q8_0` تقریباً نصف `f16` است و خروجی دو مدل روی بیشتر پیام‌ها یکسان است. برای مدل‌های کوچک، کوانتیزه‌سازی شدیدتر مثل ۴ بیت (`q4_K_M`) اثر محسوس‌تری روی کیفیت دارد. برای ساخت آن باید ابزار `llama-quantize` را از llama.cpp کامپایل کنید، یا از Space آمادهٔ [GGUF-my-repo](https://huggingface.co/spaces/ggml-org/gguf-my-repo) استفاده کنید.

قاعدهٔ کلی: هر چه مدل کوچک‌تر باشد، به کوانتیزه‌سازی حساس‌تر است. پیش از انتخاب نسخهٔ نهایی، آن را روی همان مجموعهٔ آزمون ارزیابی کنید، نه فقط با چند پیام دلخواه.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

با استفاده از تحلیل خطای بخش ۳، بخش «Limitations» در Model Card را کامل‌تر کنید: دست‌کم سه نوع پیام را نام ببرید که مدل در آن‌ها خطا دارد، با یک مثال برای هر کدام.

</div>

In [ ]:
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto"), ADAPTER_DIR)
all_data = test_data + real_data
errors = score(generate(model, conversations(all_data)), all_data).query("not all_fields")
pd.set_option("display.max_colwidth", 80)
errors[["message", "output"]].head(20)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با خواندن خطاها، بخش محدودیت‌ها را می‌توان این‌طور کامل کرد. مثال‌ها را با خطاهای واقعی که در جدول بالا می‌بینید جایگزین کنید.

</div>

In [ ]:
limitations = """## Limitations

- **Unknown products.** The model knows only the 21 catalogue products. For any other product it still
  returns one of the known ids (for example a message about a Motorola phone may come back as `P1001`).
  Always check the id against the catalogue and confirm the order with the customer.
- **Informal spelling.** Colloquial forms that were not in the synthetic training data, such as
  «واسه» instead of «برای», «دوتا» without a space, or product names written in Persian letters
  («پوکو ایکس ۷»), cause more errors.
- **Colours with spacing variants.** «نقره ای» (space) and «نقره‌ای» (zero-width non-joiner) are different
  strings; normalise both the output and the catalogue before comparing.
- **Missing fields.** When the city or colour is only implied («همون آدرس قبلی»), the model returns null.
- **Evaluation size.** Accuracy on real messages was measured on 12 hand-written examples and has a wide
  uncertainty; collect and label real messages before production use.
"""
with open("order-extractor-merged/README.md", "a", encoding="utf-8") as f:
    f.write("\n" + limitations)
print(limitations)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

مدل Ollama را روی کل ۲۰۰ پیام آزمون ارزیابی کنید و دقت را با نتیجهٔ `transformers` در بخش ۲ مقایسه کنید. اگر تفاوت هست، علتش چه می‌تواند باشد؟

</div>

In [ ]:
del model
torch.cuda.empty_cache()
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto"), ADAPTER_DIR)
hf_outputs = generate(model, conversations(test_data))
del model
torch.cuda.empty_cache()

client, name = create_ollama_model("order-extractor-q8_0.gguf", "order-extractor")
gguf_outputs = ollama_outputs(client, name, test_data)

hf, gguf = score(hf_outputs, test_data), score(gguf_outputs, test_data)
agreement = sum(parse(a) == parse(b) for a, b in zip(hf_outputs, gguf_outputs)) / len(test_data)
print(f"transformers (4-bit + adapter): {hf.all_fields.mean():.3f}")
print(f"Ollama (merged, q8_0):          {gguf.all_fields.mean():.3f}")
print(f"identical outputs: {agreement:.0%}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

دو مدل از نظر عددی یکسان نیستند و اختلاف کوچک طبیعی است. علت‌های ممکن:

- **دقت متفاوت:** در `transformers` مدل پایه ۴ بیتی NF4 است و آداپتور جدا اعمال می‌شود. در Ollama مدل ادغام‌شده با کوانتیزه‌سازی ۸ بیتی اجرا می‌شود. نسخهٔ ۸ بیتی به وزن‌های اصلی نزدیک‌تر است و ممکن است کمی **بهتر** باشد.
- **پیاده‌سازی متفاوت:** llama.cpp و PyTorch محاسبات را با ترتیب و دقت ممیز شناور متفاوتی انجام می‌دهند. در رمزگشایی حریصانه، یک تفاوت کوچک در یک توکن می‌تواند کل ادامهٔ پاسخ را عوض کند.
- **قالب گفتگو:** اگر قالب Modelfile با قالب آموزش فرق داشته باشد، اختلاف بزرگ و سیستماتیک می‌شود. اگر دقت Ollama خیلی پایین‌تر بود، اول قالب را بررسی کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

دادهٔ آموزشی را هم به شکل یک **Dataset** در Hugging Face Hub منتشر کنید (`Dataset.push_to_hub`) و یک Dataset Card بنویسید که توضیح دهد داده مصنوعی است و چطور ساخته شده است.

</div>

In [ ]:
from datasets import DatasetDict
from huggingface_hub import DatasetCard
from llm_utils import get_secret

dataset = DatasetDict({
    "train": Dataset.from_list([{"message": d["message"], **d["order"]} for d in train_data]),
    "test": Dataset.from_list([{"message": d["message"], **d["order"]} for d in test_data]),
})
card = DatasetCard("""---
language:
- fa
license: cc-by-4.0
task_categories:
- text-generation
tags:
- synthetic
- information-extraction
pretty_name: Novin Kala synthetic order messages
---

# Novin Kala synthetic order messages

Informal Persian customer messages for a fictional online store, each with the structured order
(`product_id`, `quantity`, `color`, `city`). **All messages are synthetic**: they were generated from
8 templates (plus 2 held-out templates in the test split) by `order_data.py` in the
[LLM Engineering Course](https://github.com/keyvanmahmoudi/LLM_Engineering_Course).

- train: 1,200 messages; test: 150 messages from the training templates and 50 from two new templates.
- `quantity` is 1 when the message does not mention it; `color` and `city` are null when missing.
- The store, products and prices are invented. The data does not contain personal information.
- Real customer messages are more varied; models trained only on this data generalise poorly to them.
""")

hf_token = get_secret("HF_TOKEN")
if hf_token:
    from huggingface_hub import HfApi
    repo_id = f"{HfApi(token=hf_token).whoami()['name']}/novinkala-orders"
    dataset.push_to_hub(repo_id, token=hf_token)
    card.push_to_hub(repo_id, token=hf_token)
    print("published:", f"https://huggingface.co/datasets/{repo_id}")
else:
    print(dataset)
    print("HF_TOKEN not set; skipping the upload")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

Dataset Card خوب سه چیز را صریح می‌گوید: داده **چطور ساخته شده**، یعنی مصنوعی و با الگو، **چه چیزی در آن نیست**، یعنی تنوع پیام‌های واقعی، و **مجوز** استفاده. مهم‌ترین جمله برای کسی که از این داده استفاده می‌کند، جملهٔ آخر است: مدلی که فقط روی این داده آموزش ببیند، روی پیام‌های واقعی ضعیف‌تر عمل می‌کند. پنهان کردن این محدودیت، دیگران را به نتیجه‌گیری غلط می‌رساند.

</div>